# Import Libraries

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import numpy as np
import pandas as pd
from models.tabular.XGBoost.data import get_data_reduced, get_data
from models.tabular.XGBoost.preprocess import tts, preprocess_features
from sklearn.pipeline import make_pipeline
from sklearn.metrics import recall_score, accuracy_score, f1_score, precision_score, classification_report
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from keras.callbacks import EarlyStopping

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

2026-03-18 16:54:38.775233: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-18 16:54:39.211125: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-18 16:54:44.228160: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# Load data

### Tabular data

In [3]:

# Step 1: Load data, define X and Y
df = get_data_reduced('../data')

X_tabimage = df.drop(columns=['class','gill_spacing','stem_root','stem_surface',
                                     'veil_type','veil_color','spore_print_color','scientific_name'])
y_tabimage = df['scientific_name']

# encode labels pour multi-class
le = LabelEncoder()
y_enc = le.fit_transform(y_tabimage)

# Step 2: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_tabimage, y_enc, test_size=0.3, random_state=3)
preproc= make_pipeline(preprocess_features())

# Step 3: Preprocess features
X_train_prep = preproc.fit_transform(X_train)
X_test_prep = preproc.transform(X_test)

print(f"X train and test shapes: {X_train_prep.shape}, {X_test_prep.shape}")

✅ Tabular data loaded and cleaned, shape: (61069, 17)
✅ Reduced tabular data loaded and cleaned, shape: (16944, 18)
X train and test shapes: (11860, 67), (5084, 67)


### Image data
*Don't forget to NOT take the aug_xx images*

In [ ]:
#paths = list(Path("../data/image_dataset/").rglob("*.png"))
#df_imgpath = pd.DataFrame({
#    "path": paths,
#    "label": [p.parent.name for p in paths]
#})
#df_imgpath

In [4]:

def normalize_name(name):
    return name.lower().replace("_", " ").replace("-", " ").strip()

def load_data_multiclass(data_tabular_image, batch_size=32):
    """
    Load image dataset filtered by:
    - species present in tabular data
    - excluding augmented images (prefix 'aug_')

    Returns:
        train_ds, val_ds
    """

    # 📁 Path dataset
    DATA_DIR = Path("../data/image_dataset")

    # 🎯 espèces autorisées
    allowed_species = set(data_tabular_image["scientific_name"].apply(normalize_name))

    # 🧠 mapping label
    species_to_label = {
        species: i for i, species in enumerate(sorted(allowed_species))
    }

    # 📸 récupération images
    paths = [
        p for p in DATA_DIR.rglob("*")
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    # 🔥 filtrage (espèces + exclusion aug_)
    filepaths = []
    labels = []

    for p in paths:
        # ❌ skip images augmentées
        if p.name.startswith("aug_"):
            continue

        species = normalize_name(p.parent.name)

        if species not in allowed_species:
            continue

        filepaths.append(str(p))
        labels.append(species_to_label[species])

    print(f"✅ Nombre d'images gardées : {len(filepaths)}")
    print(f"✅ Nombre de classes : {len(species_to_label)}")

    # 🔀 split stratifié
    X_train, X_val, y_train, y_val = train_test_split(
        filepaths,
        labels,
        test_size=0.2,
        stratify=labels,
        random_state=123
    )

    # 📦 création datasets TF
    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))

    # 📸 fonction de chargement image
    def load_image(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, (128, 128))
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    # ⚙️ pipeline
    train_ds = (
        train_ds
        .shuffle(buffer_size=len(X_train), seed=123)
        .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    val_ds = (
        val_ds
        .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    return train_ds, val_ds

In [6]:
train_ds, val_ds = load_data_multiclass(df)

✅ Nombre d'images gardées : 1714
✅ Nombre de classes : 48


## Implementation of the models

In [ ]:
# 0 = edible, 1 = poisonous
#mismatch_species = data_tabular_image[
#    ((data_tabular_image['type'] == 'edible') & (data_tabular_image['class'] == 1)) |
#    ((data_tabular_image['type'] == 'poisonous') & (data_tabular_image['class'] == 0))
#]['scientific_name'].unique()
#
#print("✅ Scientific names with type/class mismatch:")
#print(mismatch_species)
# lactarius deterrimus, lactarius subdulcis : in edible image (poisonous tab)
# amanita citrina, clitocybe nebularis, coprinellus micaceus : in poisonous image (edible tab)

# amanita citrina : poisonous
# clitocybe nebularis : poisonous
# coprinellus micaceus : edible
# lactarius deterrimus : edible
# lactarius subdulcis : edible

✅ Scientific names with type/class mismatch:
['Clitocybe nebularis' 'Amanita citrina' 'Coprinellus micaceus'
 'Lactarius deterrimus' 'Lactarius subdulcis']


## Image model

In [9]:
from tensorflow.keras import layers, Sequential, optimizers
from tensorflow.keras.metrics import AUC

def initialize_baseline_model_multiclass(num_classes=48, input_shape=(128, 128, 3)):
    """
    Baseline CNN for multiclass classification.
    """
    model = Sequential([
        layers.Input(shape=input_shape),

        # Convolutions
        layers.Conv2D(32, kernel_size=(4, 4), padding="same", strides=(1, 1), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPool2D(pool_size=(2, 2), padding="same"),

        layers.Conv2D(64, kernel_size=(3, 3), padding="same", strides=(1, 1), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPool2D(pool_size=(2, 2), padding="same"),

        layers.Conv2D(128, kernel_size=(3, 3), padding="same", strides=(1, 1), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPool2D(pool_size=(2, 2), padding="same"),

        layers.Conv2D(512, kernel_size=(3, 3), padding="same", activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPool2D(pool_size=(2, 2), padding="same"),

        # Dense layers
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),

        # Output layer multiclass
        layers.Dense(num_classes, activation="softmax")
    ])

    return model


def compile_model_multiclass(model):
    """
    Compile multiclass CNN.
    """
    adam = optimizers.Adam(learning_rate=1e-4)
    model.compile(
        loss="sparse_categorical_crossentropy",   # pour labels one-hot
        optimizer=adam,
        metrics=["accuracy"]  # précision et rappel peuvent être ajoutés mais nécessitent plus de setup pour multiclass
    )

    return model


In [12]:
model = initialize_baseline_model_multiclass()
model = compile_model_multiclass(model)

es = EarlyStopping(patience=10, restore_best_weights=True)

# Baseline training
history = model.fit(
     train_ds,
     batch_size=32,
     epochs=30,
     validation_data=val_ds,
     verbose=1,
     callbacks=[es])


Epoch 1/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 26s 560ms/step - accuracy: 0.0314 - loss: 4.1634 - val_accuracy: 0.0233 - val_loss: 4.5263
Epoch 2/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 24s 558ms/step - accuracy: 0.0525 - loss: 3.7845 - val_accuracy: 0.0233 - val_loss: 5.5721
Epoch 3/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 24s 563ms/step - accuracy: 0.0605 - loss: 3.7048 - val_accuracy: 0.0233 - val_loss: 6.3521
Epoch 4/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 24s 564ms/step - accuracy: 0.0729 - loss: 3.5965 - val_accuracy: 0.0233 - val_loss: 7.3174
Epoch 5/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 24s 561ms/step - accuracy: 0.1174 - loss: 3.4397 - val_accuracy: 0.0262 - val_loss: 8.2137
Epoch 6/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 25s 572ms/step - accuracy: 0.1503 - loss: 3.2247 - val_accuracy: 0.0233 - val_loss: 7.2999
Epoch 7/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 25s 580ms/step - accuracy: 0.1663 - loss: 3.0829 - val_accuracy: 0.0321 - val_loss: 6.4115
Epoch 8/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 25s 586ms/step - accuracy: 0.2137 - loss: 2.9240 - val_accu

## Tabular model

### LightGBM or XGBoost

In [ ]:
## Model xgb

model_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=3)
#model_xgb.fit(X_train_prep, y_train)
cross_val_score(model_xgb, X_train_prep, y_train, cv=5, scoring='accuracy').mean()

In [ ]:

model = LGBMClassifier(
    objective="multiclass",
    num_class=48,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

cross_val_score(model, X_train_prep, y_train, cv=5, scoring='accuracy').mean()

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000655 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 134
[LightGBM] [Info] Number of data points in the train set: 9488, number of used features: 67
[LightGBM] [Info] Start training from score -3.879668
[LightGBM] [Info] Start training from score -3.900288
[LightGBM] [Info] Start training from score -3.874579
[LightGBM] [Info] Start training from score -3.879668
[LightGBM] [Info] Start training from score -3.895093
[LightGBM] [Info] Start training from score -3.921341
[LightGBM] [Info] Start training from score -3.864478
[LightGBM] [Info] Start training from score -3.864478
[LightGBM] [Info] Start training from score -3.834773
[LightGBM] [Info] Start training from score -3.834773
[LightGBM] [Info] Start training from score -3.854478
[LightGBM] [Info] Start training from score -3.884784
[LightGBM] [Info] Start training from score -3.884784
[LightGBM] 

np.float64(0.9284148397976392)

In [ ]:
model.fit(X_train_prep, y_train)

y_pred = model.predict(X_test_prep)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000424 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 134
[LightGBM] [Info] Number of data points in the train set: 11860, number of used features: 67
[LightGBM] [Info] Start training from score -3.879668
[LightGBM] [Info] Start training from score -3.900288
[LightGBM] [Info] Start training from score -3.871538
[LightGBM] [Info] Start training from score -3.879668
[LightGBM] [Info] Start training from score -3.896130
[LightGBM] [Info] Start training from score -3.921341
[LightGBM] [Info] Start training from score -3.863474
[LightGBM] [Info] Start training from score -3.863474
[LightGBM] [Info] Start training from score -3.835749
[LightGBM] [Info] Start training from score -3.835749
[LightGBM] [Info] Start training from score -3.855474
[LightGBM] [Info] Start training from score -3.887865
